# Questão 5 - Análise de clientes
### Cenário

A Diretoria da LH Nautical deseja identificar os clientes fieis. Diferente de quem compra muito
uma única vez, o cliente fiel é o cliente que possui um gasto médio alto por transação e navega por
diversas categorias da loja. O objetivo e mapear o que esses clientes de elite estão consumindo para replicar o comportamento em outros segmentos.

### Premissas obrigatórias:

- Faturamento Total: Soma da coluna total por cliente.
- Frequência: Contagem total de transações (IDs de venda) por cliente.
- Ticket Médio: Faturamento Total / Frequência.
- Diversidade de Categorias: Quantidade de categorias distintas que o cliente comprou.
- Nota: É necessário limpar os nomes das categorias no arquivo produtos_raw.csv (ex: consolidar "Ancorajen", "Encoragem" e "Ancoragem" como uma única categoria).
- Filtro de Elite: Apenas clientes que compraram produtos de 3 ou mais categorias distintas
devem ser considerados no ranking.
- Desempate: Em caso de empate no Ticket Médio, utilize o `id_cliente` em ordem crescente.

### Tarefa

1. Realize a limpeza das categorias de produtos para evitar duplicidade por erro de grafia.
2. Calcule o Ticket Médio e a Diversidade de Categorias para cada `id_cliente`.
3. Filtre os 10 clientes com o maior Ticket Médio que atendam ao critério de diversidade (3+
categorias).
4. Para este grupo específico de 10 clientes, identifique qual categoria de produto concentra
a maior quantidade total de itens comprados `(sum(qtd))`.

In [1]:
import pandas as pd

produtos = pd.read_csv('../datasets_tratados/produtos_normalizados.csv')
vendas = pd.read_csv('../datasets_tratados/vendas_normalizadas.csv')

In [2]:
print(f"\nProdutos únicos após deduplicação: {len(produtos)}")
produtos.head()


Produtos únicos após deduplicação: 150


,name,price,id_product,actual_category
0,Transponder AIS Maré Magnum,33122.52,1,Eletrônicos
1,Transponder Furuno Marlin,13998.15,2,Eletrônicos
2,Radar Furuno Pulse Leviathan,9024.19,3,Eletrônicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,Eletrônicos
4,Piloto Automático Furuno Storm,23669.01,5,Eletrônicos


> Tarefa 2 - Calcule o ticket médio e a diversidade de categorias para cada `id_client`

In [3]:
print("Shape vendas:", vendas.shape)
vendas.head(3)

Shape vendas: (9895, 6)


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,2024-09-15
2,2,25,139,7,9475.3,2024-08-13


In [4]:
# Join vendas x produtos
vendas_produtos = vendas.merge(
    produtos[['id_product', 'actual_category']],
    left_on='id_product',
    right_on='id_product',
    how='left'
)

# Verifica se algum produto não mapeou a categoria
sem_categoria = vendas_produtos['actual_category'].isna().sum()
print(f"Transações sem categoria mapeada: {sem_categoria}")

Transações sem categoria mapeada: 0


In [5]:
# Define como vai ser exibido os números float de forma global
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [6]:
# Agrupamento por cliente
clientes_stats = vendas_produtos.groupby('id_client').agg(
    faturamento_total=('total', 'sum'),
    frequencia=('id', 'count'),
    qtd_categorias=('actual_category', 'nunique'),
    diversidade_categorias = ('actual_category', 'unique')
).reset_index()

# Cálculo do Ticket Médio
clientes_stats['ticket_medio'] = clientes_stats['faturamento_total'] / clientes_stats['frequencia']

print(f"Total de clientes com histórico de compra: {len(clientes_stats)}")
clientes_stats.head()

Total de clientes com histórico de compra: 49


,id_client,faturamento_total,frequencia,qtd_categorias,diversidade_categorias,ticket_medio
0,1,"51,092,500.05",190,3,"[Eletrônicos, Ancoragem, Propulsão]","268,907.89"
1,2,"65,652,931.35",220,3,"[Propulsão, Eletrônicos, Ancoragem]","298,422.42"
2,3,"59,575,349.10",207,3,"[Ancoragem, Propulsão, Eletrônicos]","287,803.62"
3,4,"50,691,754.40",207,3,"[Propulsão, Ancoragem, Eletrônicos]","244,887.70"
4,5,"58,592,802.70",202,3,"[Ancoragem, Eletrônicos, Propulsão]","290,063.38"


> Tarefa 3 - Filtre os 10 clientes com o maior Ticket Médio que atendam ao critério de diversidade (3+
categorias)

In [15]:
# Filtro de diversidade (3 ou mais categorias distintas)
clientes_elite = clientes_stats[clientes_stats['qtd_categorias'] >= 3].copy()

# Ordenação por Ticket Médio (descendente) e ID Cliente (ascendente para desempate)
top10_clientes = clientes_elite.sort_values(
    by=['ticket_medio', 'id_client'], 
    ascending=[False, True]
).head(10).reset_index(drop=True) 

top10_clientes.index += 1

print(" 🏆 Top 10 Clientes de Elite")
print('-' * 75)
print(top10_clientes[['id_client', 'faturamento_total', 'frequencia', 'ticket_medio', 'qtd_categorias']])
print('-' * 75)
print(f"Quantidade Total de Categorias: {len(produtos['actual_category'].value_counts())}")
print(f"Categorias: {produtos['actual_category'].unique().tolist()}")

 🏆 Top 10 Clientes de Elite
---------------------------------------------------------------------------
    id_client  faturamento_total  frequencia  ticket_medio  qtd_categorias
1          47      64,003,343.75         190    336,859.70               3
2          42      72,187,369.50         222    325,168.33               3
3           9      66,788,855.35         218    306,370.90               3
4          22      59,581,398.75         198    300,916.16               3
5           2      65,652,931.35         220    298,422.42               3
6          28      60,826,837.25         204    298,170.77               3
7          46      59,126,834.35         199    297,119.77               3
8          38      57,093,331.15         195    292,786.31               3
9          36      62,791,038.15         215    292,051.34               3
10          5      58,592,802.70         202    290,063.38               3
-----------------------------------------------------------------------

> Tarefa 4 - Para este grupo específico de 10 clientes, identifique qual categoria de produto concentra
a maior quantidade total de itens comprados `(sum(qtd))`

In [22]:
# Filtrar as vendas originais apenas para os 10 clientes de elite
vendas_elite = vendas_produtos[vendas_produtos['id_client'].isin(top10_clientes['id_client'])]

# Identificar a categoria com maior volume de itens comprados
preferencia_elite = (
    vendas_elite
    .groupby('actual_category')
    .agg(total_itens=('qtd', 'sum'))
    .sort_values(by='total_itens', ascending=False)
)

# Categoria líder
categoria_top = preferencia_elite.index[0]
quantidade_top = preferencia_elite.iloc[0]['total_itens']

print(f"Categoria dominante: {categoria_top}")
print(f"Total de itens (sum(qtd)): {quantidade_top}")
print("\nTotal de Itens (qtd)")
print('-' * 31)
print(preferencia_elite)

Categoria dominante: Propulsão
Total de itens (sum(qtd)): 6030

Total de Itens (qtd)
-------------------------------
                 total_itens
actual_category             
Propulsão               6030
Ancoragem               5632
Eletrônicos             5214


### Breve Análise

* **Identificação do Grupo de Elite**: Foram selecionados os 10 clientes com o maior Ticket Médio que compraram produtos de, pelo menos, 3 categorias distintas.
* **Análise de Consumo**: Para este grupo específico, foi calculada a soma da quantidade (sum(qtd)) por categoria para identificar o segmento mais consumido.
* **Resultados das Categorias (Grupo de Elite)**:
    * **Propulsão**: 6.030 itens comprados.
    * **Ancoragem**: 5.632 itens comprados.
    * **Eletrônicos**: 5.214 itens comprados.

**Conclusão:**
A categoria dominante entre os clientes de elite é Propulsão, com um total de 6.030 itens adquiridos pelo grupo dos 10 melhores clientes.